In [8]:
pip install openai chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.


### 로컬에서 진행할 때 되지 않는다면 절대 경로 설정하는 법
- github 절대 경로 이용하는 법 찾아볼 예정

In [ ]:
# 벡터 DB에 넣을 파일이 있는 경로
DATA_PATH = r"C:\Users\sunhe\OneDrive\문서\GitHub\Card-Recommendation-Chatbot\data\check_cards_benefits" 

indexing_existing_data()

[C:\Users\sunhe\OneDrive\문서\GitHub\Card-Recommendation-Chatbot\data\check_cards_benefits] 경로에서 고도화된 데이터를 인덱싱 중...
✅ 371개의 고도화된 데이터가 Vector DB에 등록되었습니다.


## 기존에 먼저 작성했던 코드

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# ==========================================
# 1. 초기 설정 (API 키 및 경로)
# ==========================================
os.environ["OPENAI_API_KEY"] = "사용자 API 키 값" # 본인 API 키 입력
client = OpenAI()

# 기존 데이터 경로 설정 (파일을 이미 벡터 DB에 넣었기에 생략함)
# DATA_PATH = "../data/cards/check_cards_benefits"

# 한국어 임베딩 모델 설정
ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# Vector DB 연결
chroma_client = chromadb.PersistentClient(path="./card_vector_db")

# 기존 컬렉션 삭제 (임베딩 충돌 방지 및 최신화용)
try:
    chroma_client.delete_collection(name="card_info_v2")
except:
    pass

collection = chroma_client.create_collection(
    name="card_info_v2", 
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 기존 JSON 데이터 로드 및 인덱싱
# ==========================================
def indexing_existing_data():
    file_list = [f for f in os.listdir(DATA_PATH) if f.endswith('.json')]
    
    ids, documents, metadatas = [], [], []

    print(f"[{DATA_PATH}] 경로에서 데이터를 읽어오는 중...")

    for i, file_name in enumerate(file_list):
        with open(os.path.join(DATA_PATH, file_name), 'r', encoding='utf-8') as f:
            card = json.load(f)
            
            # RAG 성능을 높이기 위해 모든 정보를 문장으로 결합 (Context 생성)
            context_text = (
                f"카드사: {card['company']}, 카드명: {card['card_name']}. "
                f"캐시백: {card.get('cashback', '정보없음')}, "
                f"주요혜택처: {card.get('benefit_place', '정보없음')}, "
                f"추가할인: {card.get('discount', '정보없음')}, "
                f"전월실적조건: {card.get('performance', '전월실적 없음')}, "
                f"해외사용: {card.get('overseas', '정보없음')}."
            )
            
            ids.append(f"card_{i}")
            documents.append(context_text)
            metadatas.append({"name": card['card_name'], "company": card['company']})

    collection.add(ids=ids, documents=documents, metadatas=metadatas)
    print(f"✅ {len(documents)}개의 카드 정보가 Vector DB에 등록되었습니다.")

# ==========================================
# 3. RAG 답변 생성 함수
# ==========================================
def get_ai_response(query, persona="", use_rag=True):
    """
    use_rag=True 이면 RAG 적용 답변, False 이면 순수 GPT 답변 (평가 비교용)
    """
    if use_rag:
        # DB에서 관련 정보 검색 (Top 3)
        results = collection.query(query_texts=[query], n_results=3)
        retrieved_context = "\n".join(results['documents'][0])
        
        system_content = f"""
        당신은 카드 추천 전문 비서입니다. 사용자의 페르소나를 고려하여 답변하세요.
        페르소나: {persona}
        
        반드시 제공된 [카드 데이터]의 내용에만 기반하여 답변하세요. 
        데이터에 없는 내용은 지어내지 말고 모른다고 답하세요.

        [카드 데이터]
        {retrieved_context}
        """
    else:
        # Base GPT-3.5 성능 확인용
        system_content = f"당신은 카드 추천 비서입니다. 페르소나({persona})에 맞춰 답변하세요."

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": query}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

# ==========================================
# 4. 평가를 위한 실행부
# ==========================================
indexing_existing_data()

# 테스트 페르소나 예시
persona_A = "사회초년생, 배달앱과 스타벅스를 자주 이용함, 실적 압박이 없는 카드를 원함"
user_query = "나에게 맞는 체크카드를 추천해줘."

print("\n" + "="*50)
print(f"실험 질문: {user_query}")
print(f"페르소나: {persona_A}")
print("="*50)

# 1) Base 모델 답변 (RAG 미적용)
print("\n[1. Base GPT-3.5 답변]")
print(get_ai_response(user_query, persona_A, use_rag=False))

# 2) 고도화 모델 답변 (RAG 적용)
print("\n[2. RAG 적용 GPT-3.5 답변]")
print(get_ai_response(user_query, persona_A, use_rag=True))

FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: '../data/cards/check_cards_benefits'

## 카드 혜택에 대한 세부 내용을 받은 후에 작성한 추가 코드

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# ==========================================
# 1. 초기 설정 (API 키 및 경로)
# ==========================================
os.environ["OPENAI_API_KEY"] = "사용자 API 키 값" # 본인 API 키 입력
client = OpenAI()

# 기존 데이터 경로 설정 (파일을 이미 벡터 DB에 넣었기에 생략함)
# DATA_PATH = "../data/cards/check_cards_benefits"

ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

chroma_client = chromadb.PersistentClient(path="./card_vector_db")

try:
    chroma_client.delete_collection(name="card_info_v3_complex")
except:
    pass

collection = chroma_client.create_collection(
    name="card_info_v3_complex", 
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 고도화된 JSON 데이터 전처리 함수
# ==========================================
def format_card_to_context(card):
    """
    benefit 리스트 내의 상세 카테고리와 내용을 
    AI가 검색하기 좋은 긴 문장으로 변환합니다.
    """
    # 기본 정보 결합
    base_text = (
        f"카드사: {card['company']}, 카드명: {card['card_name']}. "
        f"대표혜택: {card.get('cashback', '')}, {card.get('benefit_place', '')}, {card.get('discount', '')}. "
        f"전월실적조건: {card.get('performance', '없음')}. "
        f"해외사용여부: {card.get('overseas', '정보없음')}."
    )
    
    # benefit 리스트 내부의 상세 혜택 풀기
    detail_benefits = []
    if "benefit" in card and isinstance(card["benefit"], list):
        for b in card["benefit"]:
            category = b.get("category", "기타")
            content = b.get("content", "")
            detail_benefits.append(f"[{category}] {content}")
    
    # 상세 혜택이 있다면 문장에 추가
    if detail_benefits:
        detail_text = " 상세 세부혜택: " + " / ".join(detail_benefits)
        return base_text + detail_text
    
    return base_text

# ==========================================
# 3. 데이터 로드 및 인덱싱
# ==========================================
def indexing_existing_data():
    file_list = [f for f in os.listdir(DATA_PATH) if f.endswith('.json')]
    ids, documents, metadatas = [], [], []

    print(f"[{DATA_PATH}] 경로에서 고도화된 데이터를 인덱싱 중...")

    for i, file_name in enumerate(file_list):
        with open(os.path.join(DATA_PATH, file_name), 'r', encoding='utf-8') as f:
            card = json.load(f)
            
            # 전처리 함수 호출 (이 부분이 고도화의 핵심)
            context_text = format_card_to_context(card)
            
            ids.append(f"card_{i}")
            documents.append(context_text)
            metadatas.append({"name": card['card_name'], "company": card['company']})

    collection.add(ids=ids, documents=documents, metadatas=metadatas)
    print(f"✅ {len(documents)}개의 고도화된 데이터가 Vector DB에 등록되었습니다.")

# ==========================================
# 4. RAG 답변 생성 및 비교 실행부
# ==========================================
# (기존 get_ai_response 함수와 동일하게 사용 가능)
def get_ai_response(query, persona="", use_rag=True):
    if use_rag:
        results = collection.query(query_texts=[query], n_results=3)
        retrieved_context = "\n".join(results['documents'][0])
        
        system_content = f"""
        당신은 카드 고릴라 데이터를 기반으로 하는 카드 추천 전문 비서입니다.
        아래 [제공된 카드 정보]를 꼼꼼히 분석하여 답변하세요.
        
        [페르소나]
        {persona}
        
        [제공된 카드 정보]
        {retrieved_context}
        
        [답변 가이드라인]
        1. 세부 카테고리(통신, 요식 등)별 구체적인 캐시백 금액이나 %를 언급하세요.
        2. 전월실적 조건을 반드시 확인하여 답변에 포함하세요.
        3. 데이터에 없는 카드는 절대 추천하지 마세요.
        """
    else:
        system_content = f"카드 추천 비서로서 페르소나({persona})에 맞춰 답변하세요."

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": query}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

# ==========================================
# 4. 평가를 위한 실행부
# ==========================================
indexing_existing_data()

# 테스트 페르소나 예시
persona_A = "사회초년생, 배달앱과 스타벅스를 자주 이용함, 실적 압박이 없는 카드를 원함"
user_query = "나에게 맞는 체크카드를 추천해줘."

print("\n" + "="*50)
print(f"실험 질문: {user_query}")
print(f"페르소나: {persona_A}")
print("="*50)

# 1) Base 모델 답변 (RAG 미적용)
print("\n[1. Base GPT-3.5 답변]")
print(get_ai_response(user_query, persona_A, use_rag=False))

# 2) 고도화 모델 답변 (RAG 적용)
print("\n[2. RAG 적용 GPT-3.5 답변]")
print(get_ai_response(user_query, persona_A, use_rag=True))

[C:\Users\sunhe\OneDrive\문서\GitHub\Card-Recommendation-Chatbot\data\check_cards_benefits] 경로에서 고도화된 데이터를 인덱싱 중...
✅ 371개의 고도화된 데이터가 Vector DB에 등록되었습니다.

[RAG 적용 고도화 모델 답변]:
당신을 위한 카드를 추천해드리겠습니다. 

주로 마트에서 장을 보시는 분이신 것으로 보아, 대표적인 마트 관련 혜택을 제공하는 카드를 추천해드릴텐데요, 전월실적이 없는 것을 고려하여 "나마네카드"와 "핀크카드" 중에서 선택할 수 있을 것 같습니다.

1. "나마네카드"는 NAMANE 카드 리워드와 30% 소득공제를 제공합니다. 이 카드는 전월실적이 없어도 혜택을 받을 수 있으며, 나만의 카드를 제작할 수 있는 선택형 혜택도 제공합니다.

2. "핀크카드"는 이용 실적에 따라 최대 1% 적립과 연말정산 30% 공제 혜택을 제공합니다. 전월실적이 10만원 이상이어야 하지만, AI핀고의 정확한 소비 분석과 한정판 디자인 혜택을 누릴 수 있습니다.

마트에서의 소비를 고려하면, "나마네카드"의 30% 소득공제 혜택이 유용할 수 있습니다. 또한, 나만의 카드 제작이 가능하므로 자신만의 디자인을 가진 카드를 사용할 수 있습니다.


### 두 모델의 성능 평가 진행할 것, 페르소나 추가를 진행해야 함

In [ ]:
import pandas as pd

# ==========================================
# 5. 성능 평가(Evaluation) 실행부
# ==========================================

def run_evaluation_test(test_scenarios):
    evaluation_results = []

    print("🚀 모델 비교 테스트를 시작합니다...")

    for i, scene in enumerate(test_scenarios):
        persona = scene['persona']
        query = scene['query']
        
        print(f"\n[테스트 {i+1}] 페르소나: {persona[:30]}...")
        
        # 1) Base 모델 답변 생성
        base_ans = get_ai_response(query, persona, use_rag=False)
        
        # 2) RAG 모델 답변 생성
        rag_ans = get_ai_response(query, persona, use_rag=True)
        
        # 결과 저장 (사람이 나중에 점수를 매길 수 있도록 구성)
        evaluation_results.append({
            "ID": i + 1,
            "Persona": persona,
            "Query": query,
            "Base_Response": base_ans,
            "RAG_Response": rag_ans,
            "Base_Score": "", # 사람이 직접 채울 공간 (상/중/하)
            "RAG_Score": "",  # 사람이 직접 채울 공간 (상/중/하)
            "Memo": ""        # 특징 기입
        })

    # 결과를 데이터프레임으로 변환 및 CSV 저장
    df = pd.DataFrame(evaluation_results)
    df.to_csv("model_evaluation_sheet.csv", index=False, encoding='utf-8-sig')
    print("\n✅ 테스트 완료! 'model_evaluation_sheet.csv' 파일이 생성되었습니다.")
    return df

# --- 테스트 시나리오 설정 (최소 10개 이상 권장) ---
scenarios = [
    {"persona": "사회초년생, 배달앱과 스타벅스 자주 이용, 실적 압박 없는 카드 원함", "query": "나에게 맞는 체크카드 하나 추천해줘."},
    {"persona": "해외 직구를 즐기는 대학생, 수수료 면제 혜택이 중요함", "query": "해외 결제할 때 혜택 좋은 카드 알려줘."},
    {"persona": "육아 중인 부모, 병원/약국 지출이 많고 대형마트 장보기가 필수", "query": "병원비랑 마트 할인되는 카드 있을까?"},
    {"persona": "자취생, 통신비 자동이체 할인을 원하고 편의점을 매일 이용함", "query": "고정 지출인 통신비 아낄 수 있는 카드 추천해줘."},
    # ... 여기에 시나리오를 추가하여 10개 이상 만드세요.
]

# 실행
eval_df = run_evaluation_test(scenarios)

In [ ]:
def print_final_statistics(csv_file):
    df = pd.read_csv(csv_file)
    
    # 점수를 숫자로 변환 (상=3, 중=2, 하=1)
    score_map = {"상": 3, "중": 2, "하": 1}
    df['Base_Num'] = df['Base_Score'].map(score_map)
    df['RAG_Num'] = df['RAG_Score'].map(score_map)
    
    print("\n" + "="*50)
    print("🏆 최종 모델 비교 성능 통계")
    print("="*50)
    print(f"Base GPT-3.5 평균 점수: {df['Base_Num'].mean():.2f}")
    print(f"RAG 고도화 모델 평균 점수: {df['RAG_Num'].mean():.2f}")
    
    print("\n[만족도 분포]")
    print(f"Base 모델 '상' 비율: {(df['Base_Score']=='상').mean()*100:.1f}%")
    print(f"RAG 모델 '상' 비율: {(df['RAG_Score']=='상').mean()*100:.1f}%")
    print("="*50)

# 사용법: 사람이 직접 채운 CSV 파일명을 입력하세요.
# print_final_statistics("model_evaluation_sheet_filled.csv")